# 10 · Efficient multiprocessing

**Time:** 35–45 minutes. **You need:** lessons 03, 07, and 09; Python 3.11+;
NumPy in your notebook environment. See the [course setup](README.md).

You will learn to separate worker startup from repeated work, group small tasks,
and reduce repeated transfers of arrays. Our CPU example performs Python
arithmetic. Our array example sums a small NumPy array. Neither decodes images.

Run top to bottom. Predict each result before running it. All examples are local,
use two processes by default, and need no GPU.

[Course guide](README.md) · Previous: [Failures and clean shutdown](09_failures_and_clean_shutdown.ipynb)
· Next: [Native library parallelism](11_native_library_parallelism.ipynb)

**NumPy** is a library for working with arrays of numbers. **MiB** means 1,048,576 bytes; we use it when discussing array storage.

## 1. Where the time goes

A process pool does more than calculate. It starts Python interpreters, imports
modules, sends arguments, runs functions, and sends results back. Workers have
their own Python memory. Those costs can outweigh the benefit of extra CPU cores.

**Serialization** means encoding an object as bytes so another process can
reconstruct it. Python process pools normally use `pickle` for this transfer.
Sending a large array repeatedly can mean encoding, copying, and reconstructing
its data repeatedly. A small task identifier is much cheaper to send.

**Batching** groups several jobs into one worker call. It saves some scheduling
and transfer overhead. It can also leave another worker idle if one large batch
takes much longer than the others. [Process-pool arguments and results](https://docs.python.org/3.11/library/concurrent.futures.html#processpoolexecutor).

In [1]:
import inspect
import json
import subprocess
import sys
from pathlib import Path

import numpy as np

if sys.version_info < (3, 11):
    raise RuntimeError("Use Python 3.11 or later for this lesson.")
PROJECT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "lessons" / "efficient_process_demo.py").exists()),
    None,
)
if PROJECT is None:
    raise FileNotFoundError("Open this notebook from the course folder.")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from lessons import efficient_process_demo as demo
HELPER = PROJECT / "lessons" / "efficient_process_demo.py"


def run_demo(*arguments):
    completed = subprocess.run(
        [sys.executable, "-X", "utf8", str(HELPER), *map(str, arguments)],
        cwd=PROJECT, capture_output=True, text=True, encoding="utf-8", timeout=90,
    )
    if completed.returncode:
        raise RuntimeError(
            f"Helper exited with code {completed.returncode}:\n"
            + completed.stdout + "\n" + completed.stderr
        )
    if completed.stderr:
        print(completed.stderr)
    return json.loads(completed.stdout)


print("Python:", sys.version.split()[0], "NumPy:", np.__version__)

Python: 3.13.7 NumPy: 2.2.6


Process workers need functions defined in an importable file. The helper has
top-level functions and a guarded entry point: `if __name__ == "__main__":`.
It uses `spawn`, which starts a fresh interpreter, on every operating system.
The notebook launches that file with the kernel's Python interpreter.
The launcher waits for completion, shows captured output on failure, and has a 90-second timeout. A timeout stops
the helper; it does not guarantee that every child process has also stopped.
[Safe importing with spawn](https://docs.python.org/3.11/library/multiprocessing.html#the-spawn-and-forkserver-start-methods).

## 2. Use the same work for every comparison

A task is `(job_id, iterations)`. Its result is `(job_id, score)`. The same
task list goes through the sequential version and each process-pool version.
The helper checks their results for equality.

In [2]:
print(inspect.getsource(demo.cpu_score))
print("Example result:", demo.cpu_score((0, 10)))

def cpu_score(task):
    """A deterministic Python loop. Every mode runs the same tasks."""
    job_id, iterations = task
    score = 0
    for value in range(iterations):
        score += ((value + job_id) ** 2) % 97
    return job_id, score

Example result: (0, 285)


**Predict:** will a fresh pool finish 12 small jobs faster than a simple loop?
Will the answer change once those workers are already running?

We measure four things:

| Measurement | What it includes |
| --- | --- |
| Sequential | One pass through the same jobs on the main process |
| Fresh pool, end to end | Pool construction, worker startup, one pass, and shutdown |
| First map in a second pool | First submission, worker startup, and one pass; excludes final shutdown |
| Warm pool | One pass using workers that have already completed a pass; excludes startup and shutdown |

We repeat each warm setting three times and report its median, the middle value.
That reduces some noise. It does not remove differences caused by your machine
or prove how a large production workload will behave.

In [3]:
batching = run_demo("--demo", "batching")
assert batching["all_results_match"]
print(f"Sequential:            {batching['sequential_seconds']:.4f} s")
print(f"Fresh pool end to end: {batching['fresh_pool_seconds']:.4f} s")
print(f"First map, new pool:   {batching['first_map_seconds']:.4f} s")
print("\nWarm pool: median of three passes")
print("chunksize | scheduled chunks | seconds")
for row in batching["warm_pool"]:
    print(f"{row['chunksize']:9d} | {row['chunks']:16d} | {row['median_seconds']:.4f}")

Sequential:            0.0705 s
Fresh pool end to end: 0.1473 s
First map, new pool:   0.1256 s

Warm pool: median of three passes
chunksize | scheduled chunks | seconds
        1 |               12 | 0.0397
        4 |                3 | 0.0472
       12 |                1 | 0.0715


Read the results as observations. If the fresh pool is slower, check its startup
cost before deciding that processes are unsuitable. If a warm pool helps, that
benefit matters only when your application actually reuses its workers.

The helper times completed work: `list(pool.map(...))` waits for all results.
Timing only the call to `submit()` would measure submission, not completion.

## 3. Choose a chunk size

`ProcessPoolExecutor.map(..., chunksize=4)` sends groups of up to four inputs as
scheduled chunks. Your function still receives one task at a time. Increasing
`chunksize` changes how the pool schedules the jobs; it does not change the
arithmetic. **Scheduling** means deciding which worker receives the next job. [Process-pool chunking](https://docs.python.org/3.11/library/concurrent.futures.html#concurrent.futures.Executor.map).

With 12 tasks and `chunksize=12`, there is only **one scheduled chunk**. One
worker does all its tasks. Two workers cannot split that single chunk.

This option has no effect on a thread pool. Also, on Python 3.11–3.13, executor
`map` collects/submits its inputs eagerly. We use only 12 tasks here. Chunking
alone does not provide a memory limit for a huge stream; lesson 12 will bound
the number of submissions.

In [4]:
source = inspect.getsource(demo.batching_demo)
start = source.index("    rows = []")
stop = source.index("    return {", start)
print(source[start:stop])

    rows = []
    with ProcessPoolExecutor(max_workers=workers, mp_context=context) as pool:
        # The first map starts the new workers. The timer excludes final shutdown.
        first, first_seconds = timed_call(
            lambda: list(pool.map(cpu_score, tasks, chunksize=1)))
        assert first == expected
        for chunksize in dict.fromkeys((1, 4, jobs)):
            measurements = []
            for _ in range(3):
                results, seconds = timed_call(
                    lambda: list(pool.map(cpu_score, tasks, chunksize=chunksize)))
                assert results == expected
                measurements.append(seconds)
            rows.append({"chunksize": chunksize,
                         "chunks": (jobs + chunksize - 1) // chunksize,
                         "median_seconds": median(measurements)})




Notice the `with` around the repeated measurements. All of those passes use
one pool. Each `assert` checks correctness before we trust the timing.

**Predict:** the last quarter of the next task list takes four times as much
work per task. What might happen when those slow jobs are grouped together?

In the excerpt, `timed_call(function)` returns the function's result and its elapsed seconds. `measurements` holds three durations, and `median(...)` selects their middle value.

In [5]:
uneven = run_demo("--demo", "batching", "--uneven")
assert uneven["all_results_match"]
for row in uneven["warm_pool"]:
    print(f"chunksize={row['chunksize']:2d}: {row['median_seconds']:.4f} s")

chunksize= 1: 0.0830 s
chunksize= 4: 0.1076 s
chunksize=12: 0.1348 s


Smaller chunks let idle workers pick up remaining work more often. Larger chunks
reduce scheduling overhead but give the pool fewer chances to distribute jobs.
This is a tradeoff, not a rule that the smallest chunk must win.

Try `--iterations 240000` with the same job count. Change only one setting,
rerun, and explain whether startup now accounts for a smaller share of the work.

## 4. Inspect the size of a transferred object

A NumPy **view** can refer to an existing array without copying its data locally.
Passing that view to a process pool normally serializes its values. A view in
the parent does not automatically become shared memory in the worker.

**Predict:** which needs more serialized bytes: an array of 262,144 integers,
or a tuple containing a shared block's name and the slice boundaries?

`np.int64` stores each integer in eight bytes. A **dtype** describes an array's
number format, such as these 64-bit integers.

In [6]:
import pickle
from time import perf_counter

small_array = np.arange(262_144, dtype=np.int64)  # 2 MiB of numeric values.
example_descriptor = ("example-block-name", small_array.shape,
                      small_array.dtype.str, 0, len(small_array))
for label, value in [("array", small_array), ("descriptor", example_descriptor)]:
    start = perf_counter()
    encoded = pickle.dumps(value, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"{label:10s}: {len(encoded):,} bytes; encoding {perf_counter() - start:.6f} s")
del small_array, encoded

array     : 2,097,291 bytes; encoding 0.000349 s
descriptor: 56 bytes; encoding 0.000095 s


The descriptor tells a worker **where to look**. It contains no array values.
The real block must exist and remain alive while workers use it.

These timings measure local encoding only. End-to-end transfer also involves
pipes, operating-system buffers, decoding, and scheduling. Do not label
`pickle.dumps()` time as total transfer time.

## 5. Share a read-only numeric array

`multiprocessing.shared_memory.SharedMemory` creates a named memory block that
several processes can attach to. A NumPy array can use that block as its storage.
[Shared memory with NumPy](https://docs.python.org/3.11/library/multiprocessing.shared_memory.html).

Our parent copies the array into the block once. Workers receive small
descriptors, read different slices, and return ordinary integers. We keep the
parent array unchanged and mark every shared view as read-only. This avoids
concurrent writes in the lesson. The flag applies to that NumPy view; it is
not operating-system protection against another process writing to the block.

In [7]:
source = inspect.getsource(demo.shared_demo)
start = source.index("    setup_start =")
stop = source.index("        # Pickling here", start)
print(source[start:stop])

    setup_start = perf_counter()
    owner = shared_memory.SharedMemory(create=True, size=array.nbytes)
    shared_view = None
    try:
        shared_view = np.ndarray(array.shape, dtype=array.dtype, buffer=owner.buf)
        shared_view[:] = array  # One initial copy into the shared block.
        shared_view.flags.writeable = False
        descriptors = [(owner.name, array.shape, array.dtype.str,
                        int(edges[i]), int(edges[i + 1])) for i in range(workers)]
        setup_seconds = perf_counter() - setup_start




A descriptor is `(name, shape, dtype, start, stop)`:

- `name` identifies the existing block.
- `shape` and `dtype` explain how to interpret its bytes as numbers.
- `start` and `stop` identify the slice for this worker.

Worker code attaches to the block, calculates, and closes its handle in `finally`.

A **handle** is this process's connection to the block. `np.ndarray(..., buffer=handle.buf)` builds an array view over the block rather than copying its numbers. Setting `view.flags.writeable = False` prevents writes through that view.

In [8]:
print(inspect.getsource(demo.sum_shared_slice))

def sum_shared_slice(descriptor):
    """Attach, read a slice, and close this worker's handle, even on failure."""
    import numpy as np
    name, shape, dtype, start, stop = descriptor
    handle = shared_memory.SharedMemory(name=name)
    view = None
    try:
        view = np.ndarray(shape, dtype=np.dtype(dtype), buffer=handle.buf)
        view.flags.writeable = False  # No worker in this lesson writes shared data.
        return int(np.sum(view[start:stop], dtype=np.int64))
    finally:
        del view  # Drop our array view before closing its backing handle.
        handle.close()



**Predict:** will sharing a 2 MiB array be faster for every application?
Consider the initial copy, the cost of attaching, and how often the array is reused.

The next experiment runs both methods in the same pool after an initial pass.
It alternates their order across three passes. Every pass must return the same
per-slice sums. The array has only 2 MiB; startup may still be the largest cost.

In [9]:
shared = run_demo("--demo", "shared", "--mib", 2)
assert shared["all_results_match"]
print(f"Whole array:                  {shared['array_bytes'] / 1024**2:.1f} MiB")
print(f"Pickled one-worker slice:     {shared['serialized_slice_bytes']:,} bytes")
print(f"Pickled one-worker descriptor:{shared['serialized_descriptor_bytes']:,} bytes")
print(f"Create block and initial copy:{shared['shared_setup_seconds']:.6f} s")
print(f"First array pass in new pool: {shared['first_array_map_seconds']:.6f} s")
print(f"Warm array transfer, median:  {shared['warm_array_seconds']:.6f} s")
print(f"Warm shared reads, median:    {shared['warm_shared_seconds']:.6f} s")

Whole array:                  2.0 MiB
Pickled one-worker slice:     1,048,715 bytes
Pickled one-worker descriptor:51 bytes
Create block and initial copy:0.000558 s
First array pass in new pool: 0.143051 s
Warm array transfer, median:  0.002121 s
Warm shared reads, median:    0.000512 s


The warm shared timing includes sending descriptors, attaching and closing
worker handles, reading slices, and returning sums. It excludes the initial
block setup, worker startup, and final pool shutdown. The warm array timing
includes sending slices and returning sums, with the same exclusions.

A single shared pass must also pay the reported setup cost. Repeated use can
spread that cost over more passes. Our sum is a NumPy operation; it is not
evidence about GIL limits for a Python loop or proof that parallel sums help.
Shared memory also keeps the original array **and** its shared copy alive in
this demonstration. Reduced repeated transfer does not mean zero memory cost.

## 6. Give cleanup a clear owner

Each worker closes its own handle. The parent waits for the pool to finish,
releases its array view, closes its handle, and calls `unlink()` once in `finally`.
That cleanup also runs when the body raises an exception. The inner `finally`
still attempts `unlink()` if `close()` raises an error.

`close()` releases a handle. `unlink()` removes the named block on platforms
that use it. On Windows, the block disappears once all handles have closed;
`unlink()` has no effect there. [Cleanup behavior](https://docs.python.org/3.11/library/multiprocessing.shared_memory.html#multiprocessing.shared_memory.SharedMemory.close).

In [10]:
source = inspect.getsource(demo.shared_demo)
print("Parent's final cleanup:")
print(source[source.index("    finally:\n        del shared_view"):])

Parent's final cleanup:
    finally:
        del shared_view
        try:
            owner.close()
        finally:
            # Try to remove the name even if closing the handle raises an error.
            # On Windows, closing all handles removes the block; unlink does nothing.
            owner.unlink()



If workers need to write, define who owns each region or coordinate access.
Two workers updating the same numbers can create a race condition. A faster
transfer method does not make those updates safe.

## 7. Practice

1. Run the batching experiment with `--workers 1`. Why can larger chunks still
   change runtime when there is no parallel calculation?
2. Run the shared experiment with `--mib 1`, then `--mib 4`. Compare serialized
   sizes and warm timings. Explain why the descriptor stays small.
3. You process a million small records with a few unusually slow records. Why
   might one enormous chunk be a poor choice?
4. An array is used only once. What costs belong in an honest comparison between
   ordinary transfer and shared memory?

In [11]:
# Practice cell: change one option, run, and explain what you observe.
# one_worker = run_demo("--demo", "batching", "--workers", 1)
# four_mib = run_demo("--demo", "shared", "--mib", 4)

<details>
<summary>Worked answers</summary>

1. One worker still pays for scheduling and exchanging messages. Grouping work
   may reduce those costs. It cannot make two CPU jobs execute in parallel.
2. Array bytes increase fourfold from 1 to 4 MiB. A descriptor holds a name,
   dimensions, a type, and two boundaries; its size barely changes. Timings
   may vary, so explain the observed measurements rather than promise a ratio.
3. One chunk gives all records to one worker. With fewer large chunks, slow
   records grouped together can leave another worker idle. Try modest chunks
   and measure the whole application.
4. Include block creation, copying into it, attaching, computing, result
   transfer, handle cleanup, and worker startup/shutdown when those are paid
   by the application. Include ordinary serialization and transfer on the
   other side. Keep the computational work and worker lifetime comparable.

</details>

You should now be able to explain why worker count, chunk size, pool lifetime,
and transfer size all matter. In the next lesson, you will inspect threads
started inside numeric libraries.

Next: [Native library parallelism](11_native_library_parallelism.ipynb).